# Long-Horizon Planning Analysis

This notebook analyzes the long-horizon planning capabilities of world models, including planning horizon estimation, belief drift measurement, and imagination fidelity analysis.

## Overview
- Estimate the model's effective planning horizon
- Measure belief drift over long rollouts
- Analyze imagination fidelity against real observations
- Compare planning across different environment types

In [ ]:
# Setup and imports
import torch
import numpy as np
from world_model_lens.patching.advanced_patching import estimate_planning_horizon, belief_drift_rollout
from world_model_lens import HookedWorldModel
from world_model_lens.utils.trajectory_dataset import TrajectoryDataset
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)

print("Long-horizon analysis environment setup complete")

## Load Model and Data

Load a world model and trajectory data for long-horizon analysis.

In [ ]:
# Load world model
model = HookedWorldModel.from_pretrained("dreamerv3 Atari")
model.eval()

# Load trajectory dataset
dataset = TrajectoryDataset(
    env_name="Atari/Pong",
    num_trajectories=100,
    trajectory_length=1000,
    include_rewards=True,
    include_dones=True
)

print(f"Model loaded: DreamerV3 Atari")
print(f"Dataset: {len(dataset)} trajectories, length {dataset.trajectory_length}")
print(f"Observation shape: {dataset.observation_dim}")
print(f"Action space: {dataset.action_dim} dimensions")

## Estimate Planning Horizon

Estimate how far into the future the model can effectively plan by measuring prediction accuracy degradation over different rollout lengths.

In [ ]:
# Configure planning horizon estimation
from world_model_lens.patching.advanced_patching import HorizonEstimationConfig

horizon_config = HorizonEstimationConfig(
    horizon_range=np.arange(1, 101, 5),  # Test horizons from 1 to 100
    num_evaluation_samples=50,
    metric="mse",  # or "kl_divergence", "ssim"
    bootstrap_samples=10,
    confidence_level=0.95
)

print("Estimating planning horizon...")
horizon_results = estimate_planning_horizon(
    model=model,
    dataset=dataset,
    config=horizon_config
)

print("\nEstimation complete!")
print(f"Estimated effective horizon: {horizon_results['effective_horizon']:.1f} steps")
print(f"Horizon at 80% accuracy: {horizon_results['horizon_80_percent']} steps")

In [ ]:
# Visualize horizon estimation results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Prediction accuracy vs horizon
ax1 = axes[0]
horizons = horizon_results["horizons"]
accuracies = horizon_results["accuracies"]
ci_lower = horizon_results["ci_lower"]
ci_upper = horizon_results["ci_upper"]

ax1.plot(horizons, accuracies, "b-", linewidth=2, label="Mean Accuracy")
ax1.fill_between(horizons, ci_lower, ci_upper, alpha=0.3, color="blue")
ax1.axhline(y=0.8, color="red", linestyle="--", label="80% threshold")
ax1.axvline(x=horizon_results['effective_horizon'], color="green", linestyle=":", 
            label=f"Effective horizon: {horizon_results['effective_horizon']:.1f}")
ax1.set_xlabel("Planning Horizon (steps)")
ax1.set_ylabel("Prediction Accuracy")
ax1.set_title("Prediction Accuracy vs Planning Horizon")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy degradation rate
ax2 = axes[1]
degradation_rate = horizon_results["degradation_rate"]
ax2.bar(["Short-term", "Mid-term", "Long-term"], 
        [degradation_rate["short"], degradation_rate["mid"], degradation_rate["long"]],
        color=["#2ecc71", "#f39c12", "#e74c3c"])
ax2.set_ylabel("Accuracy Loss per 10 Steps")
ax2.set_title("Degradation Rate by Time Scale")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("planning_horizon.png", dpi=150)
plt.show()

## Measure Belief Drift

Track how the model's belief state diverges from reality over long rollouts.

In [ ]:
# Configure belief drift measurement
from world_model_lens.patching.advanced_patching import BeliefDriftConfig

drift_config = BeliefDriftConfig(
    rollout_length=200,
    drift_metrics=["kl_divergence", "euclidean_distance", "cosine_similarity"],
    state_components=["mean", "std", "hidden"],
    measure_frequency=10
)

print("Measuring belief drift over long rollouts...")
drift_results = belief_drift_rollout(
    model=model,
    dataset=dataset,
    config=drift_config,
    num_rollouts=20
)

print(f"\nBelief drift measurement complete!")
print(f"Mean KL divergence at horizon: {drift_results['kl_at_horizon']:.4f}")
print(f"Drift rate: {drift_results['drift_rate']:.4f} per step")

In [ ]:
# Visualize belief drift
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# KL divergence over time
ax1 = axes[0, 0]
steps = drift_results["steps"]
ax1.plot(steps, drift_results["kl_divergence"], "b-", linewidth=2)
ax1.fill_between(steps, 
                 drift_results["kl_divergence"] - drift_results["kl_std"],
                 drift_results["kl_divergence"] + drift_results["kl_std"],
                 alpha=0.3)
ax1.set_xlabel("Rollout Step")
ax1.set_ylabel("KL Divergence")
ax1.set_title("Belief State KL Divergence Over Time")
ax1.grid(True, alpha=0.3)

# Euclidean distance
ax2 = axes[0, 1]
ax2.plot(steps, drift_results["euclidean_distance"], "g-", linewidth=2)
ax2.set_xlabel("Rollout Step")
ax2.set_ylabel("Euclidean Distance")
ax2.set_title("State Representation Distance Over Time")
ax2.grid(True, alpha=0.3)

# Cosine similarity
ax3 = axes[1, 0]
ax3.plot(steps, drift_results["cosine_similarity"], "r-", linewidth=2)
ax3.set_xlabel("Rollout Step")
ax3.set_ylabel("Cosine Similarity")
ax3.set_title("State Representation Cosine Similarity")
ax3.grid(True, alpha=0.3)

# Drift rate by component
ax4 = axes[1, 1]
components = list(drift_results["component_drift_rates"].keys())
rates = list(drift_results["component_drift_rates"].values())
colors = plt.cm.viridis(np.linspace(0, 1, len(components)))
ax4.bar(components, rates, color=colors)
ax4.set_xlabel("State Component")
ax4.set_ylabel("Drift Rate")
ax4.set_title("Drift Rate by State Component")
ax4.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("belief_drift.png", dpi=150)
plt.show()

## Analyze Imagination Fidelity

Compare the model's imagined trajectories against real observations to assess imagination quality.

In [ ]:
# Run imagination fidelity analysis
from world_model_lens.patching.advanced_patching import imagination_fidelity_analysis

fidelity_config = {
    "num_samples": 30,
    "horizon": 50,
    "metrics": ["ssim", "mse", "perceptual_similarity"],
    "aggregate_by": ["time_step", "reward_regime"]
}

print("Analyzing imagination fidelity...")
fidelity_results = imagination_fidelity_analysis(
    model=model,
    dataset=dataset,
    config=fidelity_config
)

print(f"\nImagination fidelity analysis complete!")
print(f"Mean SSIM: {fidelity_results['mean_ssim']:.4f}")
print(f"Mean MSE: {fidelity_results['mean_mse']:.6f}")

In [ ]:
# Visualize imagination fidelity
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# SSIM over horizon
ax1 = axes[0, 0]
time_steps = fidelity_results["time_steps"]
ax1.plot(time_steps, fidelity_results["ssim_by_time"], "b-", linewidth=2, label="SSIM")
ax1.fill_between(time_steps, 
                  fidelity_results["ssim_by_time"] - fidelity_results["ssim_std"],
                  fidelity_results["ssim_by_time"] + fidelity_results["ssim_std"],
                  alpha=0.3)
ax1.set_xlabel("Time Step")
ax1.set_ylabel("SSIM")
ax1.set_title("Structural Similarity Over Imagination Horizon")
ax1.grid(True, alpha=0.3)

# MSE over horizon
ax2 = axes[0, 1]
ax2.plot(time_steps, fidelity_results["mse_by_time"], "r-", linewidth=2)
ax2.set_xlabel("Time Step")
ax2.set_ylabel("MSE")
ax2.set_title("Mean Squared Error Over Imagination Horizon")
ax2.grid(True, alpha=0.3)

# Fidelity by reward regime
ax3 = axes[1, 0]
regimes = fidelity_results["fidelity_by_reward"]
regime_names = list(regimes.keys())
ssim_values = [regimes[r]["ssim"] for r in regime_names]
ax3.bar(regime_names, ssim_values, color=["#2ecc71", "#f39c12", "#e74c3c"])
ax3.set_xlabel("Reward Regime")
ax3.set_ylabel("Mean SSIM")
ax3.set_title("Imagination Fidelity by Reward Regime")
ax3.tick_params(axis="x", rotation=45)

# Sample images: real vs imagined
ax4 = axes[1, 1]
# Create a simple comparison visualization
sample_idx = 5
real_img = fidelity_results["sample_comparisons"][sample_idx]["real"]
imagined_imgs = fidelity_results["sample_comparisons"][sample_idx]["imagined"]

# Show first imagined frame
ax4.imshow(imagined_imgs[0].reshape(64, 64, 3) if imagined_imgs[0].size == 64*64*3 else np.zeros((64, 64, 3)))
ax4.set_title("Sample Imagined Frame (t=0)")
ax4.axis("off")

plt.tight_layout()
plt.savefig("imagination_fidelity.png", dpi=150)
plt.show()

## Compare Across Environment Types

Analyze how planning capabilities vary across different environment types.

In [ ]:
# Compare planning across multiple environments
from world_model_lens.patching.advanced_patching import compare_environments

environments = ["Atari/Pong", "Atari/Breakout", "DeepMind Lab/corridor"]

print("Comparing planning across environments...")
comparison_results = compare_environments(
    model=model,
    environments=environments,
    num_trajectories=20
)

print("\nEnvironment Comparison:")
print("="*60)
for env, results in comparison_results.items():
    print(f"\n{env}:")
    print(f"  Effective horizon: {results['effective_horizon']:.1f} steps")
    print(f"  Belief drift rate: {results['drift_rate']:.4f}")
    print(f"  Imagination SSIM: {results['mean_ssim']:.4f}")

In [ ]:
# Visualize environment comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

env_names = list(comparison_results.keys())

# Effective horizon
ax1 = axes[0]
horizons = [comparison_results[e]["effective_horizon"] for e in env_names]
ax1.bar(env_names, horizons, color="#3498db")
ax1.set_xlabel("Environment")
ax1.set_ylabel("Effective Horizon (steps)")
ax1.set_title("Planning Horizon by Environment")
ax1.tick_params(axis="x", rotation=45)

# Belief drift rate
ax2 = axes[1]
drift_rates = [comparison_results[e]["drift_rate"] for e in env_names]
ax2.bar(env_names, drift_rates, color="#e74c3c")
ax2.set_xlabel("Environment")
ax2.set_ylabel("Drift Rate")
ax2.set_title("Belief Drift Rate by Environment")
ax2.tick_params(axis="x", rotation=45)

# Imagination fidelity
ax3 = axes[2]
ssims = [comparison_results[e]["mean_ssim"] for e in env_names]
ax3.bar(env_names, ssims, color="#2ecc71")
ax3.set_xlabel("Environment")
ax3.set_ylabel("Mean SSIM")
ax3.set_title("Imagination Fidelity by Environment")
ax3.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("environment_comparison.png", dpi=150)
plt.show()

## Save Results

Export all analysis results.

In [ ]:
import json

# Compile all results
all_results = {
    "planning_horizon": {
        "effective_horizon": float(horizon_results["effective_horizon"]),
        "horizon_80_percent": int(horizon_results["horizon_80_percent"]),
        "degradation_rate": {k: float(v) for k, v in horizon_results["degradation_rate"].items()}
    },
    "belief_drift": {
        "kl_at_horizon": float(drift_results["kl_at_horizon"]),
        "drift_rate": float(drift_results["drift_rate"]),
        "component_drift_rates": {k: float(v) for k, v in drift_results["component_drift_rates"].items()}
    },
    "imagination_fidelity": {
        "mean_ssim": float(fidelity_results["mean_ssim"]),
        "mean_mse": float(fidelity_results["mean_mse"])
    },
    "environment_comparison": {
        env: {
            "effective_horizon": float(r["effective_horizon"]),
            "drift_rate": float(r["drift_rate"]),
            "mean_ssim": float(r["mean_ssim"])
        }
        for env, r in comparison_results.items()
    }
}

with open("long_horizon_analysis.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("Results saved to long_horizon_analysis.json")